# CTB ProSiT reproduction (local Jupyter)
Run all cells. The notebook first exposes the saved models, then reproduces the final baseline and what-if results.

In [ ]:
%pip install -q -r requirements.txt

## 1. Load and inspect the saved models
PNML stores the common control flow. JSON provides a readable ProSiT export. PKL preserves the exact calibrated runtime objects used in the thesis.

In [ ]:
from pathlib import Path
from IPython.display import display
import reproduce

ROOT = Path.cwd()
if not (ROOT / 'models').is_dir():
    raise FileNotFoundError('Open the notebook from the CTB handover folder.')

json_models = reproduce.load_json_models()
print('JSON models loaded with SimulatorParameters.from_json():')
display(reproduce.inspect_models(json_models))

In [ ]:
pickle_models = reproduce.load_pickle_models()
print('Exact thesis model objects loaded from PKL:')
display(reproduce.inspect_models(pickle_models))

print('Intervention checks:')
display(reproduce.check_model_changes(pickle_models))

## 2. Run the simulations
The full setting runs 10 matched seeds × 3 models × 17,892 cases. Set `RUN_FULL = False` only for a short mechanics test.

In [ ]:
RUN_FULL = True
output_dir = reproduce.run(full=RUN_FULL)
print(f'Fresh result tables: {output_dir}')

## 3. Compare with the thesis results
This is an exact technical reproduction check. The scientific scenario effects are contained in `scenario_paired_delta_summary.csv`.

In [ ]:
if RUN_FULL:
    comparison = reproduce.compare(output_dir)
    display(comparison)
    print('FULL REPRODUCTION PASSED')
else:
    print('Smoke test passed; thesis values are compared only after a full run.')

## Interpretation
The paired comparison uses the same seeds for each intervention and the baseline. T22 closure produces no resolved overall performance effect in this abstraction. The 20% demand intervention raises the realised arrival rate and produces a small resolved increase in RMG service time, while turnaround and pre-service effects remain unresolved. These are model-conditional findings, not direct physical causal estimates for the terminal.